# 🔑 Solutions: Lesson 5: Feature Selection
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)


from sklearn.impute import SimpleImputer

fe = build_farm_features(load_lesson_clean())          # Lesson-04 recipes (see utils/course_utils.py)

NUM = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "phosphorus_ppm", "potassium_ppm",
       "organic_matter_pct", "fertilizer_kg_ha", "farm_size_ha", "irrigation", "plant_month", "plant_year",
       "grow_days", "npk_total", "n_to_p_ratio", "moisture_index", "fert_x_irrig", "log_fert", "log_farm_size",
       "rain_gap_sq", "temp_gap_sq", "ph_gap_sq", "pest_ord", "exp_ord"]

# median-impute (for this lesson only - on the whole data; the pipeline version comes in Lesson 7)
X_num = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(fe[NUM]), columns=NUM, index=fe.index)
crops = pd.get_dummies(fe["crop"], prefix="crop", dtype=float)

rng = np.random.default_rng(0)
canaries = pd.DataFrame(rng.normal(size=(len(fe), 3)), columns=["noise_1", "noise_2", "noise_3"], index=fe.index)

X = pd.concat([X_num, crops, canaries], axis=1)
X["temp_f"] = X["avg_temp_c"] * 9 / 5 + 32               # a 'helpful colleague' added Fahrenheit...
y = np.log(fe["yield_t_ha"])                              # regression target
y_cls = (fe["yield_t_ha"] > fe.groupby("crop")["yield_t_ha"].transform("median")).astype(int)   # 'above-median for its crop'

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
spectra = load_spectra()

---
### Problem 5.1 &nbsp; ⭐ &nbsp; Prune redundant features

Build the feature matrix `X` (see the first code cell of the lesson - it's in the solution setup too).

1. List every pair of features with absolute correlation **above 0.9**.
2. Write a function `drop_correlated(X, threshold)` that returns the names to drop, keeping the *first* feature of each correlated pair.
3. How many features are removed? Is the Fahrenheit column among them?

In [2]:
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

pairs = upper.stack()
pairs = pairs[pairs > 0.9].sort_values(ascending=False).round(3)
print("Highly correlated pairs:\n", pairs.to_string())

def drop_correlated(X, threshold=0.9):
    c = X.corr().abs()
    u = c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    return [col for col in u.columns if (u[col] > threshold).any()]

to_drop = drop_correlated(X)
print("\nDropped:", to_drop, "| temp_f dropped?", "temp_f" in to_drop)

Highly correlated pairs:
 avg_temp_c   temp_f            1.000
rainfall_mm  moisture_index    0.902

Dropped: ['moisture_index', 'temp_f'] | temp_f dropped? True


**💬 Why this works:** Taking the upper triangle avoids counting each pair twice. Dropping the *later* column of each pair keeps one representative. A perfectly correlated duplicate (Fahrenheit vs Celsius) carries zero new information. Which of two correlated features to keep is a judgement call: prefer the more interpretable or cheaper one.

---
### Problem 5.2 &nbsp; ⭐⭐ &nbsp; The canary test

Add **five** random-noise columns (`rng = np.random.default_rng(1)`, standard normal) to the feature matrix `X`. Rank *all* features by:
(a) the F-statistic (`f_regression`) and (b) mutual information (`mutual_info_regression`, `random_state=0`).

Report (1) the ranks of the five noise columns for each method and (2) the **noise floor**: the best score any canary achieved, and how many *real* features score above it. What would you conclude about a real feature scoring below the floor?

In [3]:
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

rng2 = np.random.default_rng(1)
Xc = X.copy()
for i in range(5):
    Xc[f"canary_{i}"] = rng2.normal(size=len(Xc))

F  = pd.Series(SelectKBest(f_regression, k="all").fit(Xc, y).scores_, index=Xc.columns)
MI = pd.Series(mutual_info_regression(Xc, y, random_state=0), index=Xc.columns)

ranks = pd.DataFrame({"F rank": F.rank(ascending=False), "MI rank": MI.rank(ascending=False)}).astype(int)
print(f"{Xc.shape[1]} features in total. Canary ranks:")
canaries_ = [c for c in Xc.columns if c.startswith("canary")]
print(ranks.loc[canaries_])

real = [c for c in Xc.columns if c not in canaries_]
for name, sc in (("F-test", F), ("mutual info", MI)):
    floor = sc[canaries_].max()
    print(f"{name:12s} noise floor = {floor:8.3f} → {int((sc[real] > floor).sum())} of {len(real)} real features score above it")

38 features in total. Canary ranks:
          F rank  MI rank
canary_0      15       20
canary_1      38       33
canary_2      19       33
canary_3      28       25
canary_4      36       17
F-test       noise floor =   10.203 → 14 of 33 real features score above it
mutual info  noise floor =    0.019 → 16 of 33 real features score above it


**💬 Why this works:** Random columns cannot truly predict anything, but by luck they still score a little, and they land somewhere in the middle of the weak features. The best canary score is your **noise floor**: real features below it can't be told apart from chance, features far above it are credible. Canaries are a free sanity check for *any* selection method.

---
### Problem 5.3 &nbsp; ⭐⭐ &nbsp; How many features do we really need? (RFECV)

Use `RFECV` with a *pipeline* of `StandardScaler → Ridge` (so scaling happens inside each CV fold) and 5-fold CV on `X`, `y`.

Report the optimal number of features and list the selected ones. Are the three `noise_*` columns and `temp_f` selected?

In [4]:
from sklearn.feature_selection import RFECV

pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
sel = RFECV(pipe, step=1, cv=5, scoring="r2", importance_getter="named_steps.ridge.coef_", n_jobs=-1).fit(X, y)

chosen = X.columns[sel.support_].tolist()
print("optimal number of features:", sel.n_features_)
print("selected:", chosen)
print("noise / temp_f selected?:", [c for c in chosen if c.startswith("noise") or c == "temp_f"] or "none")

optimal number of features: 19
selected: ['rainfall_mm', 'nitrogen_ppm', 'organic_matter_pct', 'fertilizer_kg_ha', 'irrigation', 'plant_year', 'npk_total', 'n_to_p_ratio', 'moisture_index', 'log_fert', 'rain_gap_sq', 'temp_gap_sq', 'ph_gap_sq', 'pest_ord', 'exp_ord', 'crop_Beans', 'crop_Maize', 'crop_Potato', 'crop_Sorghum']
noise / temp_f selected?: none


**💬 Why this works:** RFECV repeatedly fits, removes the weakest feature and measures cross-validated score, then keeps the size that scored best. Putting the scaler *inside* the pipeline means each fold scales using only its own training rows: selection without leakage.

---
### Problem 5.4 &nbsp; ⭐⭐ &nbsp; Impurity importance vs permutation importance

Split 75/25 (`random_state=0`). Fit a `RandomForestRegressor(200, min_samples_leaf=3, random_state=0)`.

Compare for the three `noise_*` columns: the **impurity importance** (`feature_importances_`) and the **permutation importance on the test set**, each expressed as a percentage of that method's *top* feature. Which method is more fooled by noise?

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
rf = RandomForestRegressor(200, min_samples_leaf=3, n_jobs=-1, random_state=0).fit(X_tr, y_tr)

impurity = pd.Series(rf.feature_importances_, index=X.columns)
perm = pd.Series(permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0, n_jobs=-1).importances_mean, index=X.columns)

noise = ["noise_1", "noise_2", "noise_3"]
out = pd.DataFrame({"impurity (% of top)": impurity / impurity.max() * 100,
                    "permutation (% of top)": perm / perm.max() * 100}).loc[noise].round(2)
print(out)

         impurity (% of top)  permutation (% of top)
noise_1                 0.74                   -0.03
noise_2                 0.69                    0.06
noise_3                 0.79                    0.02


**💬 Why this works:** Impurity importance is computed on *training* data and gives continuous noise features some credit simply because trees can split on them to memorise noise. Permutation importance measures the damage on *unseen* data when a feature is shuffled; noise features can't hurt there, so they sit near zero (even slightly negative).

---
### Problem 5.5 &nbsp; ⭐⭐⭐ &nbsp; Principal components on soil spectra

Load the soil spectra (`load_spectra()`); the features are the 40 `band_*` columns, the target `organic_carbon_pct`.

1. Standardise the bands and fit PCA. How many components are needed to explain **90%, 95% and 99%** of the variance?
2. With 5-fold CV and a pipeline `StandardScaler → PCA(k) → Ridge`, compute R² for k = 1…8. Where does it plateau?
3. Compare with Ridge on all 40 raw bands. What did PCA buy us?

In [6]:
from sklearn.decomposition import PCA

bands = spectra.filter(like="band_")
target = spectra["organic_carbon_pct"]

cum = PCA().fit(StandardScaler().fit_transform(bands)).explained_variance_ratio_.cumsum()
for t in (0.90, 0.95, 0.99):
    print(f"{t:.0%} variance needs {int(np.argmax(cum >= t)) + 1} components")

scores = {k: cross_val_score(make_pipeline(StandardScaler(), PCA(k), Ridge()), bands, target, cv=5, scoring="r2").mean()
          for k in range(1, 9)}
print("\nCV R² by number of components:\n", pd.Series(scores).round(3).to_string())
print("\nRidge on all 40 raw bands:", round(cross_val_score(make_pipeline(StandardScaler(), Ridge()), bands, target, cv=5, scoring="r2").mean(), 3))

90% variance needs 4 components
95% variance needs 4 components
99% variance needs 14 components



CV R² by number of components:
 1   -0.007
2    0.116
3    0.785
4    0.873
5    0.873
6    0.872
7    0.872
8    0.873

Ridge on all 40 raw bands: 0.865


**💬 Why this works:** Neighbouring wavelengths are almost copies of each other, so a handful of components capture nearly all the information. Four numbers do the job of forty: a simpler, faster and less noise-sensitive model with similar accuracy. The price: components are blends of bands, harder to interpret.